# 05 — Predictive Modeling

This notebook trains and compares multiple models for predicting `Outcome`:

1. Logistic Regression
2. Regularized Logistic Regression
3. Random Forest
4. Gradient Boosting

The workflow is leakage-safe: patient identifiers are never used as predictors, repeated IDs are kept together in train/test splits, and all imputation, scaling, and one-hot encoding are performed inside scikit-learn `Pipeline` objects during cross-validation and final model fitting.

Outputs:

- `reports/model_comparison.csv`
- `models/best_model.joblib`

In [1]:
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=UserWarning)

# Resolve paths robustly whether the notebook is run from the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'CardiacPatientData.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import add_clinical_features

DATA_PATH = PROJECT_ROOT / 'data' / 'CardiacPatientData.csv'
REPORTS_DIR = PROJECT_ROOT / 'reports'
MODELS_DIR = PROJECT_ROOT / 'models'
MODEL_COMPARISON_PATH = REPORTS_DIR / 'model_comparison.csv'
BEST_MODEL_PATH = MODELS_DIR / 'best_model.joblib'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
POS_LABEL = 1

## Load data

The raw dataset is loaded before splitting. Deterministic row-wise clinical features are added only after the train/test split so no train-set population statistics can leak into the held-out test set.

In [2]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
display(df.head())

outcome_distribution = df['Outcome'].value_counts(normalize=True).sort_index().to_frame('proportion')
outcome_distribution['count'] = df['Outcome'].value_counts().sort_index()
display(outcome_distribution)

Dataset shape: 5,906 rows x 20 columns


,ID,SBP,DBP,HR,RR,BT,SpO2,Age,Gender,GCS,Na,K,Cl,Urea,Ceratinine,Alcoholic,Smoke,FHCD,TriageScore,Outcome
0,1,163,95,90,18,98,98,66,1,15,139.0,4.0,105.0,41.0,91.0,1.0,1.0,0.0,3.0,1
1,1,134,85,85,15,98,98,66,1,15,139.0,4.0,105.0,41.0,91.0,1.0,1.0,0.0,3.0,1
2,1,121,77,80,19,98,98,66,1,15,139.0,4.0,105.0,41.0,91.0,1.0,1.0,0.0,3.0,1
3,1,103,78,70,16,98,98,66,1,15,139.0,4.0,105.0,41.0,91.0,1.0,1.0,0.0,3.0,1
4,1,96,70,59,13,98,98,66,1,15,139.0,4.0,105.0,41.0,91.0,1.0,1.0,0.0,3.0,1


,proportion,count
Outcome,,
0,0.144429,853
1,0.855571,5053


## Leakage-safe train/test split

Repeated `ID` values indicate multiple rows per patient. If IDs repeat, a `StratifiedGroupKFold` split is used so all observations for the same patient stay entirely in either train or test. If IDs do not repeat, the notebook falls back to a standard stratified split.

In [3]:
id_counts = df['ID'].value_counts()
ids_repeat = bool((id_counts > 1).any())
print(f'Unique IDs: {df["ID"].nunique():,}')
print(f'Rows: {len(df):,}')
print(f'Any repeated ID values? {ids_repeat}')

X_raw = df.drop(columns=['Outcome'])
y = df['Outcome'].astype(int)

if ids_repeat:
    holdout_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    train_idx, test_idx = next(holdout_splitter.split(X_raw, y, groups=df['ID']))
    split_strategy = 'StratifiedGroupKFold by patient ID'
else:
    train_idx, test_idx = train_test_split(
        np.arange(len(df)),
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    split_strategy = 'Stratified train/test split'

train_raw = df.iloc[train_idx].copy()
test_raw = df.iloc[test_idx].copy()
overlapping_ids = set(train_raw['ID']).intersection(set(test_raw['ID']))

print(f'Split strategy: {split_strategy}')
print(f'Train rows: {len(train_raw):,}; test rows: {len(test_raw):,}')
print(f'Train unique IDs: {train_raw["ID"].nunique():,}; test unique IDs: {test_raw["ID"].nunique():,}')
print(f'Overlapping IDs across train/test: {len(overlapping_ids):,}')

display(pd.concat(
    {
        'train': train_raw['Outcome'].value_counts(normalize=True).sort_index(),
        'test': test_raw['Outcome'].value_counts(normalize=True).sort_index(),
    }, axis=1
).rename_axis('Outcome'))

Unique IDs: 112
Rows: 5,906
Any repeated ID values? True
Split strategy: StratifiedGroupKFold by patient ID
Train rows: 4,746; test rows: 1,160
Train unique IDs: 90; test unique IDs: 22
Overlapping IDs across train/test: 0


,train,test
Outcome,,
0,0.148757,0.126724
1,0.851243,0.873276


## Feature engineering and preprocessing

`ID` is removed from the modeling matrix to avoid memorization. Numerical and categorical preprocessing is defined inside a `ColumnTransformer`, which is then embedded in every model pipeline. This means imputation, scaling, and encoding are refit independently inside each cross-validation fold.

In [4]:
train_features = add_clinical_features(train_raw.drop(columns=['Outcome']))
test_features = add_clinical_features(test_raw.drop(columns=['Outcome']))

y_train = train_raw['Outcome'].astype(int)
y_test = test_raw['Outcome'].astype(int)
groups_train = train_raw['ID']

X_train = train_features.drop(columns=['ID'])
X_test = test_features.drop(columns=['ID'])

categorical_features = [
    'Gender',
    'Alcoholic',
    'Smoke',
    'FHCD',
    'TriageScore',
    'age_band',
    'gcs_severity',
    'hypoxemia_flag',
    'sodium_abnormal_flag',
    'potassium_abnormal_flag',
    'chloride_abnormal_flag',
    'urea_abnormal_flag',
    'creatinine_abnormal_flag',
]
categorical_features = [col for col in categorical_features if col in X_train.columns]
numeric_features = [col for col in X_train.columns if col not in categorical_features]

X_train_model = X_train.copy()
X_test_model = X_test.copy()
for col in categorical_features:
    X_train_model[col] = X_train_model[col].astype('object').where(X_train_model[col].notna(), np.nan)
    X_test_model[col] = X_test_model[col].astype('object').where(X_test_model[col].notna(), np.nan)

numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ],
    sparse_threshold=0.0,
)

print(f'Numeric features ({len(numeric_features)}): {numeric_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

Numeric features (15): ['SBP', 'DBP', 'HR', 'RR', 'BT', 'SpO2', 'Age', 'GCS', 'Na', 'K', 'Cl', 'Urea', 'Ceratinine', 'pulse_pressure', 'shock_index']
Categorical features (13): ['Gender', 'Alcoholic', 'Smoke', 'FHCD', 'TriageScore', 'age_band', 'gcs_severity', 'hypoxemia_flag', 'sodium_abnormal_flag', 'potassium_abnormal_flag', 'chloride_abnormal_flag', 'urea_abnormal_flag', 'creatinine_abnormal_flag']


## Cross-validation and hyperparameter tuning

The same group-aware logic is used for cross-validation: if repeated IDs exist, `StratifiedGroupKFold` is used and `groups=ID` is passed into every grid search. Models are selected by mean cross-validated AUROC, while additional metrics are computed on the held-out test set.

In [5]:
if ids_repeat:
    cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_strategy = 'StratifiedGroupKFold by patient ID'
else:
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_strategy = 'StratifiedKFold'

scoring = {
    'auroc': 'roc_auc',
    'auprc': 'average_precision',
    'f1': make_scorer(f1_score, zero_division=0),
    'sensitivity': make_scorer(recall_score, zero_division=0),
    'precision': make_scorer(precision_score, zero_division=0),
}

model_specs = {
    'Logistic Regression': {
        'pipeline': Pipeline([
            ('preprocess', preprocessor),
            ('model', LogisticRegression(C=np.inf, solver='lbfgs', max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        'param_grid': {
            'model__class_weight': [None, 'balanced'],
        },
    },
    'Regularized Logistic Regression': {
        'pipeline': Pipeline([
            ('preprocess', preprocessor),
            ('model', LogisticRegression(solver='saga', max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        'param_grid': {
            'model__C': [0.1, 1.0, 10.0],
            'model__l1_ratio': [0.0, 0.5, 1.0],
            'model__class_weight': [None, 'balanced'],
        },
    },
    'Random Forest': {
        'pipeline': Pipeline([
            ('preprocess', preprocessor),
            ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        'param_grid': {
            'model__n_estimators': [200],
            'model__max_depth': [None, 8],
            'model__min_samples_leaf': [1, 5],
            'model__class_weight': [None],
        },
    },
    'Gradient Boosting': {
        'pipeline': Pipeline([
            ('preprocess', preprocessor),
            ('model', GradientBoostingClassifier(random_state=RANDOM_STATE)),
        ]),
        'param_grid': {
            'model__n_estimators': [100],
            'model__learning_rate': [0.03, 0.1],
            'model__max_depth': [2, 3],
            'model__subsample': [1.0],
        },
    },
}

print(f'CV strategy: {cv_strategy}')

CV strategy: StratifiedGroupKFold by patient ID


In [6]:
def calculate_binary_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'test_auroc': roc_auc_score(y_true, y_score),
        'test_auprc': average_precision_score(y_true, y_score),
        'test_f1': f1_score(y_true, y_pred, zero_division=0),
        'test_sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'test_specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'test_precision': precision_score(y_true, y_pred, zero_division=0),
        'test_npv': tn / (tn + fn) if (tn + fn) else np.nan,
        'true_negative': tn,
        'false_positive': fp,
        'false_negative': fn,
        'true_positive': tp,
    }

results = []
fitted_searches = {}

for model_name, spec in model_specs.items():
    print(f'\nTuning {model_name}...')
    search = GridSearchCV(
        estimator=spec['pipeline'],
        param_grid=spec['param_grid'],
        scoring=scoring,
        refit='auroc',
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
        error_score='raise',
    )
    fit_kwargs = {'groups': groups_train} if ids_repeat else {}
    search.fit(X_train_model, y_train, **fit_kwargs)
    fitted_searches[model_name] = search

    best_estimator = search.best_estimator_
    y_score = best_estimator.predict_proba(X_test_model)[:, 1]
    y_pred = (y_score >= 0.5).astype(int)
    test_metrics = calculate_binary_metrics(y_test, y_pred, y_score)

    row = {
        'model': model_name,
        'selection_metric': 'cv_mean_auroc',
        'cv_strategy': cv_strategy,
        'train_test_split_strategy': split_strategy,
        'train_rows': len(train_raw),
        'test_rows': len(test_raw),
        'train_unique_ids': train_raw['ID'].nunique(),
        'test_unique_ids': test_raw['ID'].nunique(),
        'overlapping_train_test_ids': len(overlapping_ids),
        'best_params': search.best_params_,
        'cv_mean_auroc': search.best_score_,
        'cv_mean_auprc': search.cv_results_['mean_test_auprc'][search.best_index_],
        'cv_mean_f1': search.cv_results_['mean_test_f1'][search.best_index_],
        'cv_mean_sensitivity': search.cv_results_['mean_test_sensitivity'][search.best_index_],
        'cv_mean_precision': search.cv_results_['mean_test_precision'][search.best_index_],
    }
    row.update(test_metrics)
    results.append(row)
    print(f'Best CV AUROC: {search.best_score_:.3f}; held-out AUROC: {test_metrics["test_auroc"]:.3f}')

comparison_df = pd.DataFrame(results).sort_values(
    by=['test_auroc', 'test_auprc', 'test_f1'], ascending=False
).reset_index(drop=True)
comparison_df['rank_by_test_auroc'] = np.arange(1, len(comparison_df) + 1)
comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)

print(f'Saved model comparison table to: {MODEL_COMPARISON_PATH.relative_to(PROJECT_ROOT)}')
display(comparison_df)


Tuning Logistic Regression...


Best CV AUROC: 0.688; held-out AUROC: 0.852

Tuning Regularized Logistic Regression...


Best CV AUROC: 0.841; held-out AUROC: 0.855

Tuning Random Forest...


Best CV AUROC: 0.821; held-out AUROC: 0.881

Tuning Gradient Boosting...


Best CV AUROC: 0.798; held-out AUROC: 0.860
Saved model comparison table to: reports/model_comparison.csv


,model,selection_metric,cv_strategy,train_test_split_strategy,train_rows,test_rows,train_unique_ids,test_unique_ids,overlapping_train_test_ids,best_params,...,test_f1,test_sensitivity,test_specificity,test_precision,test_npv,true_negative,false_positive,false_negative,true_positive,rank_by_test_auroc
0,Random Forest,cv_mean_auroc,StratifiedGroupKFold by patient ID,StratifiedGroupKFold by patient ID,4746,1160,90,22,0,"{'model__class_weight': None, 'model__max_dept...",...,0.939098,0.997038,0.129252,0.887522,0.863636,19,128,3,1010,1
1,Gradient Boosting,cv_mean_auroc,StratifiedGroupKFold by patient ID,StratifiedGroupKFold by patient ID,4746,1160,90,22,0,"{'model__learning_rate': 0.03, 'model__max_dep...",...,0.926187,0.972359,0.122449,0.884201,0.391304,18,129,28,985,2
2,Regularized Logistic Regression,cv_mean_auroc,StratifiedGroupKFold by patient ID,StratifiedGroupKFold by patient ID,4746,1160,90,22,0,"{'model__C': 0.1, 'model__class_weight': 'bala...",...,0.896381,0.892399,0.319728,0.900398,0.301282,47,100,109,904,3
3,Logistic Regression,cv_mean_auroc,StratifiedGroupKFold by patient ID,StratifiedGroupKFold by patient ID,4746,1160,90,22,0,{'model__class_weight': 'balanced'},...,0.911824,0.898322,0.503401,0.925738,0.418079,74,73,103,910,4


## Save the best model

The best model is selected by held-out AUROC, with AUPRC and F1 used as tie-breakers. The saved object is the complete scikit-learn pipeline, so future scoring applies the same preprocessing steps before prediction.

In [7]:
best_model_name = comparison_df.loc[0, 'model']
best_model = fitted_searches[best_model_name].best_estimator_
joblib.dump(best_model, BEST_MODEL_PATH)

print(f'Best model: {best_model_name}')
print(f'Saved best model pipeline to: {BEST_MODEL_PATH.relative_to(PROJECT_ROOT)}')
print('Best model held-out metrics:')
display(comparison_df.loc[0, [
    'model',
    'test_auroc',
    'test_auprc',
    'test_f1',
    'test_sensitivity',
    'test_specificity',
    'test_precision',
    'test_npv',
    'best_params',
]].to_frame('value'))

Best model: Random Forest
Saved best model pipeline to: models/best_model.joblib
Best model held-out metrics:


,value
model,Random Forest
test_auroc,0.880895
test_auprc,0.980215
test_f1,0.939098
test_sensitivity,0.997038
test_specificity,0.129252
test_precision,0.887522
test_npv,0.863636
best_params,"{'model__class_weight': None, 'model__max_dept..."


## Interpretation: model performance and complexity

The cell below prints a short interpretation from the saved comparison table. In this workflow, added model complexity is considered justified only when a more complex model materially improves held-out AUROC/AUPRC over the simpler logistic baselines and does not sacrifice clinically important sensitivity or NPV.

In [8]:
best = comparison_df.iloc[0]
logistic_best = comparison_df[comparison_df['model'].isin(['Logistic Regression', 'Regularized Logistic Regression'])].iloc[0]
auroc_gain = best['test_auroc'] - logistic_best['test_auroc']
auprc_gain = best['test_auprc'] - logistic_best['test_auprc']

if best['model'] == logistic_best['model']:
    complexity_statement = (
        'The best-performing model is a logistic baseline, so additional tree-based complexity is not justified by this hold-out evaluation.'
    )
elif auroc_gain >= 0.02 or auprc_gain >= 0.02:
    complexity_statement = (
        'The added complexity appears justified because the best non-logistic model improves AUROC or AUPRC by at least 0.02 over the best logistic baseline.'
    )
else:
    complexity_statement = (
        'The added complexity is only weakly justified because the improvement over the best logistic baseline is small (<0.02 AUROC and AUPRC).'
    )

print(
    f"Best model: {best['model']} "
    f"(AUROC={best['test_auroc']:.3f}, AUPRC={best['test_auprc']:.3f}, "
    f"F1={best['test_f1']:.3f}, sensitivity={best['test_sensitivity']:.3f}, "
    f"specificity={best['test_specificity']:.3f}, precision={best['test_precision']:.3f}, "
    f"NPV={best['test_npv']:.3f})."
)
print(
    f"Best logistic baseline: {logistic_best['model']} "
    f"(AUROC={logistic_best['test_auroc']:.3f}, AUPRC={logistic_best['test_auprc']:.3f})."
)
print(f"Difference vs best logistic baseline: AUROC={auroc_gain:.3f}, AUPRC={auprc_gain:.3f}.")
print(complexity_statement)

Best model: Random Forest (AUROC=0.881, AUPRC=0.980, F1=0.939, sensitivity=0.997, specificity=0.129, precision=0.888, NPV=0.864).
Best logistic baseline: Regularized Logistic Regression (AUROC=0.855, AUPRC=0.978).
Difference vs best logistic baseline: AUROC=0.025, AUPRC=0.002.
The added complexity appears justified because the best non-logistic model improves AUROC or AUPRC by at least 0.02 over the best logistic baseline.


## Results summary

Random Forest performed best on the held-out test set by AUROC (0.881) and AUPRC (0.980). It also achieved the highest F1 score (0.939), very high sensitivity (0.997), precision of 0.888, and NPV of 0.864. The trade-off is low specificity (0.129), meaning the model favors detecting positive `Outcome` cases at the cost of more false positives.

Compared with the best logistic baseline, Regularized Logistic Regression, Random Forest improved held-out AUROC by 0.025 and AUPRC by 0.002. This suggests the added complexity is moderately justified if the main goal is ranking/discrimination and maximizing sensitivity, but the modest AUPRC gain and reduced interpretability mean the simpler regularized logistic model remains a reasonable alternative when transparency is prioritized.
